# Import patch đã duyệt theo từng ZIP
Notebook này lập chỉ mục và kiểm patch PNG đã cắt sẵn. Cell đầu kiểm kê chỉ đọc danh mục ZIP; cell smoke giải mã hai patch mỗi vật kính. Bật RUN_FULL_IMPORT khi sẵn sàng xử lý đủ 25 ZIP. Mỗi ZIP được commit riêng trên Drive để chạy lại sau khi Colab ngắt.
Bước này chưa tạo feature vectors hoặc train mô hình. Không cắt, lọc hay chỉnh màu patch.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
import shutil
import subprocess
import sys
import zipfile
from importlib import import_module
from pathlib import Path

# Copy the approved runtime bundle and the source ZIPs to Drive before starting.
RUNTIME_ZIP = Path('/content/drive/MyDrive/histology/runtime/histology-data-runtime.zip')
METADATA = Path('/content/drive/MyDrive/histology/source/Metadata.xlsx')
ARCHIVE_DIR = Path('/content/drive/MyDrive/histology/source/archives')
OUTPUT_ROOT = Path('/content/drive/MyDrive/histology/precut/v001')
WORK_ROOT = Path('/content/histology-work')  # One current ZIP is staged here.
EXPECTED_ARCHIVES = 25
RUN_FULL_IMPORT = False  # Keep false for preflight; set true after reviewing smoke output.
MAX_NEW_ZIPS = 1      # Use 1 for one-ZIP benchmark; set None for all remaining ZIPs.


In [ ]:
assert RUNTIME_ZIP.is_file(), f'Missing runtime bundle: {RUNTIME_ZIP}'
assert METADATA.is_file(), f'Missing metadata: {METADATA}'
archives = sorted(ARCHIVE_DIR.glob('Tiles-*.zip'))
assert len(archives) == EXPECTED_ARCHIVES, f'Expected {EXPECTED_ARCHIVES} ZIPs, found {len(archives)}'
runtime_dir = Path('/content/histology-data-runtime')
shutil.rmtree(runtime_dir, ignore_errors=True)
runtime_dir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(RUNTIME_ZIP) as bundle:
    bundle.extractall(runtime_dir)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                str(runtime_dir / 'requirements-data.txt')], check=True)
sys.path.insert(0, str(runtime_dir))
data_io = import_module('histology_data.io')
precut = import_module('histology_data.precut')
atomic_json = data_io.atomic_json
file_hash = data_io.file_hash
audit_import = precut.audit_import
build_inventory = precut.build_inventory
import_selected = precut.import_selected
inspect_inventory = precut.inspect_inventory
load_inventory = precut.load_inventory
write_inventory = precut.write_inventory
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f'Found {len(archives)} ZIPs; output is {OUTPUT_ROOT}')

In [ ]:
inventory_path = OUTPUT_ROOT / 'inventory.json'
if inventory_path.exists():
    inventory = load_inventory(inventory_path)
    assert inventory['metadata_sha256'] == file_hash(METADATA), 'Metadata changed; rebuild inventory.'
    print('Reusing verified inventory', inventory['inventory_id'])
else:
    inventory = build_inventory(METADATA, archives)
    write_inventory(inventory_path, inventory)
    print('Created inventory', inventory['inventory_id'])
print('Patch counts by lens:', inventory['coverage'])
print('Coordinate collisions:', inventory['coordinate_collision_count'])
print('Unmatched patch paths:', inventory['unmatched_png_count'])
assert inventory['unmatched_png_count'] == 0, 'Review unmatched patch paths before import.'
assert inventory['conflicting_candidate_case_label_count'] == 0, 'Review case-level label conflicts.'

In [ ]:
smoke_path = OUTPUT_ROOT / 'smoke_qc.json'
smoke = inspect_inventory(inventory, per_lens=2)
atomic_json(smoke_path, smoke)
print(f"Decoded {smoke['selected_patch_count']} sample patches: {smoke['selected_by_lens']}")
print('Coordinate-collision samples checked:', sum(len(group) for group in smoke['coordinate_collision_checks']))
print('Training-ready:', smoke['training_ready'])
assert smoke['selected_patch_count'] == 6

In [ ]:
if not RUN_FULL_IMPORT:
    print('Preflight only. Review smoke_qc.json, then set RUN_FULL_IMPORT=True for ZIP processing.')
else:
    result = import_selected(inventory, OUTPUT_ROOT / 'parts', max_new_parts=MAX_NEW_ZIPS, work_root=WORK_ROOT)
    print(f"Imported {result['new_parts']} new ZIPs; verified/reused {result['reused_parts']} existing ZIPs.")
    print('Run this cell again after a disconnect to verify commits and continue.')

In [ ]:
if RUN_FULL_IMPORT and len(list((OUTPUT_ROOT / 'parts' / inventory['inventory_id']).glob('*/commit.json'))) == len(archives):
    audit_path = OUTPUT_ROOT / 'dataset_audit.json'
    audit = audit_import(inventory, OUTPUT_ROOT / 'parts', audit_path, scratch_root=WORK_ROOT)
    print('Verified parts:', audit['verified_zip_parts'], '/', audit['expected_zip_parts'])
    print('Global exact duplicate groups:', audit['exact_duplicate_groups'])
    print('Training-ready:', audit['training_ready'])
else:
    print('Full-release audit waits until all ZIP parts have committed.')